# Card Data Pipeline

In [3]:
from typing import Dict
import pyarrow as pa
import time
import csv
from multiprocessing import Pool
import spacy
from loky import get_reusable_executor
from process_row import process_row, init_worker
from datasets import load_dataset_builder
from datasets import Dataset
from functools import partial
from datasets import load_dataset
from datasets import Features, Value
import json
import random
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

import bz2
from collections import Counter
from tqdm.notebook import tqdm
import os
import re
import pickle
import glob

import multiprocessing as mp
mp.set_start_method("fork", force=True)

csv.field_size_limit(sys.maxsize)

HEADERS = {"User-Agent": "derenrich@wikimedia.org -- gacha topic importer"}

VIEW_TOTAL_FILE = "view_totals.csv"
prefix = ""
CSV_ARTICLE_FILE = "articles.csv"
CSV_ARTICLE_FILE_WITH_TOPICS = prefix + "articles_with_topics.csv"
CSV_ARTICLE_FILE_WITH_SENTENCE_SPLITS = prefix +  "articles_with_sentences.csv"
CSV_ARTICLE_FILE_WITH_CAT_FILTER = "articles_with_cat_filter.csv"
CSV_ARTICLE_FILE_WITH_ATTRIB = prefix + "articles_with_attrib.csv"

PAGEVIEWS_BZ2 = "pageviews-*-user.bz2"

TOP_N = 100_000


In [2]:
BANNED_PREFIXES = set(['User', 'Talk', 'File', 'Category', 'Template', 'Wikipedia', 'Draft', 'Portal', 'TimedText', 'Module', 'MediaWiki', 'Help', 'Category', 'Event', 'Special', 'Media'])
BANNED_PREFIXES_WITH_TALK = set([])
for p in BANNED_PREFIXES:
    BANNED_PREFIXES_WITH_TALK.add(p)
    BANNED_PREFIXES_WITH_TALK.add(f"{p}_talk")

### Fetch view data

In [7]:
def count_file(view_file):
    local = Counter()
    with bz2.open(view_file, "rb") as f:
        for l in f:
            l = l.strip()
            if not l.startswith(b"en.wikipedia"):
                continue
            try:
                proj, title, pageid, plat, daily_tots, hourly = l.split()
            except ValueError:
                continue
            title = title.decode().strip('"').replace('\\"', '"')
            if not title:
                continue
            ns = title.split(":", 1)
            if len(ns) > 1 and ns[0] in BANNED_PREFIXES_WITH_TALK:
                continue
            local[title] += int(daily_tots)
    return local

# process page view data
views = Counter()
# get these from https://dumps.wikimedia.org/other/pageview_complete/monthly/2026/2026-04/pageviews-202604-user.bz2
view_files = glob.glob(PAGEVIEWS_BZ2)

views = Counter()
view_files = glob.glob(PAGEVIEWS_BZ2)
with Pool(processes=12) as pool:
    for partial in tqdm(pool.imap_unordered(count_file, view_files), total=len(view_files), desc="files"):
        views.update(partial)
with open(VIEW_TOTAL_FILE, "w") as f:
    for title, count in views.most_common():
        f.write(f"{title}\t{count}\n")


# or just read it from a file
# views = Counter()
# with open(VIEW_TOTAL_FILE, 'r') as f:
#     i = 0
#     for r in f.readlines():
#         try:
#             title, count = r.strip().split("\t")
#             views[title] = int(count)
#         except Exception as e:
#             print(e, i, "failed to get view data", r.strip())
#         i += 1


not enough values to unpack (expected 2, got 1) 10353656 failed to get view data 65


### Fetch article data and process

In [ ]:
def stream_rows():
    ds = load_dataset(
        "wikimedia/structured-wikipedia",
        "enwiki_namespace_0",
        split="train",
        streaming=True,
    )
    for row in ds:
        yield row

hit = False

with open(CSV_ARTICLE_FILE, "w", newline="") as f:
    writer = None
    with Pool(processes=4, initializer=init_worker, initargs=(views,)) as pool:
        for out in tqdm(
            pool.imap_unordered(process_row, stream_rows(), chunksize=1000),
            desc="Processing",
            unit=" rows",
        ):
            if writer is None:
                break
                writer = csv.DictWriter(f, fieldnames=out.keys())
                writer.writeheader()
            writer.writerow(out)

not enough values to unpack (expected 2, got 1) 10353656 failed to get view data 65


Processing: 0 rows [00:00, ? rows/s]

Resolving data files:   0%|          | 0/86 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/86 [00:00<?, ?it/s]

{'abstract': '"Khan gizi" spring is a spring located next to Khan\'s daughter Natevan\'s palace in the Çöl Qala neighborhood of Shusha. The spring was built in the 19th century by the order of Khurshidbanu Natavan. After the occupation of Shusha on May 8, 1992, the spring was neglected. As a result, the water of the spring has dried up, and it has fallen into neglect. Spring was included in the list of immovable historical and cultural monuments of local importance by the decision No. 132 issued by the Cabinet of Ministers of the Republic of Azerbaijan on August 2, 2001. In 2020, after the liberation of Shusha during the Second Karabakh War, the spring was repaired and returned to people\'s use.', 'additional_entities': [{'aspects': ['C.P625', 'D.en', 'O', 'S', 'T'], 'identifier': 'Q113996550', 'url': 'https://www.wikidata.org/entity/Q113996550'}], 'date_created': None, 'date_modified': datetime.datetime(2024, 10, 31, 12, 58, 40), 'description': 'Spring in Shusha, Azerbaijan', 'event':

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



### Fetch Topics (for top N)

In [5]:
TOP_N = 100_000
INFERENCE_URL = "https://api.wikimedia.org/service/lw/inference/v1/models/outlink-topic-model:predict"
HEADERS = {"User-Agent": "derenrich@wikimedia.org -- gacha topic importer"}
MAX_WORKERS = 20
MAX_RETRIES = 5
INITIAL_BACKOFF = 1.0

def get_topic(name: str) -> str:
    """Return top topic for an article name, or '' after exhausting retries."""
    page_title = name.replace(" ", "_")
    payload = json.dumps({"page_title": page_title, "lang": "en", "threshold": 0.0})
    backoff = INITIAL_BACKOFF

    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.post(INFERENCE_URL, headers=HEADERS, data=payload, timeout=30)
            # Retry on rate limit / transient server errors
            if resp.status_code in (429, 500, 502, 503, 504):
                # Honor Retry-After if provided, else exponential backoff with jitter
                retry_after = resp.headers.get("Retry-After")
                if retry_after:
                    try:
                        sleep_for = float(retry_after)
                    except ValueError:
                        sleep_for = backoff
                else:
                    sleep_for = backoff + random.uniform(0, backoff * 0.5)
                time.sleep(min(sleep_for, 120))
                backoff = min(backoff * 2, 60)
                continue

            resp.raise_for_status()
            results = resp.json().get("prediction", {}).get("results", [])
            if not results:
                return ""
            return max(results, key=lambda r: r["score"])["topic"]

        except (requests.ConnectionError, requests.Timeout):
            time.sleep(backoff + random.uniform(0, backoff * 0.5))
            backoff = min(backoff * 2, 60)
            continue
        except Exception:
            return ""

    return ""  # exhausted retries

print("loading articles")
with open(CSV_ARTICLE_FILE) as f:
    rows = list(csv.DictReader(f))

rows.sort(key=lambda r: int(r.get("view_count") or 0), reverse=True)

# deduplicate rows
dedup_rows = []
seen_urls = set()

for r in tqdm(rows, desc="deduping"):
    if r['url'] in seen_urls:
        continue
    seen_urls.add(r['url'])
    dedup_rows.append(r)
rows = dedup_rows

# drop rows without images
rows = [r for r in rows if r['image_url'] is not None and r['image_url'] != ""]

# 2. Pick top N for API enrichment
to_predict = rows[:TOP_N]

print("starting fetching")

# 3. Parallel API calls with progress bar
topics = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(get_topic, r["name"]): r["name"] for r in to_predict}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Fetching topics"):
        name = futures[fut]
        topics[name] = fut.result()

# 4. Write augmented CSV
fieldnames = list(rows[0].keys()) + ["topic", "percentile"]
with open(CSV_ARTICLE_FILE_WITH_TOPICS, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for idx, r in enumerate(rows):
        r["topic"] = topics.get(r["name"], "")
        r['percentile'] = idx / len(rows)
        writer.writerow(r)
print("done")

loading articles


deduping:   0%|          | 0/7597149 [00:00<?, ?it/s]

starting fetching


Fetching topics:   0%|          | 0/100000 [00:00<?, ?it/s]

done


### Extract sentences

In [4]:
csv.field_size_limit(sys.maxsize)

NUM_SENTENCES = 4

# Load spaCy with only the sentencizer for speed — no need for full pipeline
# have to have to run `python -m spacy download en_core_web_sm`
nlp = spacy.load("en_core_web_sm", disable=["ner", "tagger", "lemmatizer", "attribute_ruler"])

# Count rows for progress bar
with open(CSV_ARTICLE_FILE_WITH_TOPICS) as f:
    total = sum(1 for _ in f) - 1

with open(CSV_ARTICLE_FILE_WITH_TOPICS, newline="") as f_in, open(CSV_ARTICLE_FILE_WITH_SENTENCE_SPLITS, "w", newline="") as f_out:
    reader = csv.DictReader(f_in)
    sentence_fields = [f"sentence_{i+1}" for i in range(NUM_SENTENCES)]
    fieldnames = reader.fieldnames + sentence_fields
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()

    # Stream (abstract, row) pairs so we can re-attach the row after parsing
    def stream():
        i = 0
        for row in reader:
            if not row.get("topic"):
                continue
            yield (row.get("abstract") or "", row)
            i += 1
            if i > TOP_N:
                break

    # nlp.pipe with as_tuples passes context through unchanged
    pipe = nlp.pipe(
        stream(),
        as_tuples=True,
        n_process=13,
        batch_size=400,
    )

    for doc, row in tqdm(pipe, total=total, desc="Splitting sentences"):
        sentences = [s.text.strip() for s in doc.sents][:NUM_SENTENCES]
        for i, field in enumerate(sentence_fields):
            row[field] = sentences[i] if i < len(sentences) else ""
        writer.writerow(row)


Splitting sentences:   0%|          | 0/2763586 [00:00<?, ?it/s]

### Fetch attribution information

In [5]:
URL = "https://en.wikipedia.org/w/rest.php/attribution/v0-beta/pages/File:{}/signals"
HEADERS = {"User-Agent": "derenrich@wikimedia.org - gacha attribution data"}

MAX_WORKERS = 4
MAX_RETRIES = 5
INITIAL_BACKOFF = 1.0

def fetch_attribution(image_url):
    if not image_url:
        return {}
    image_filename = image_url.split("/")[-1]
    if not image_filename:
        return {}
    
    backoff = INITIAL_BACKOFF
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(URL.format(image_filename), headers=HEADERS, timeout=10)
            if resp.status_code in (429, 500, 502, 503, 504):
                retry_after = resp.headers.get("Retry-After")
                sleep_for = float(retry_after) if retry_after else backoff + random.uniform(0, backoff * 0.1)
                time.sleep(sleep_for)
                backoff = min(backoff * 2, 60)
                continue
            resp.raise_for_status()
            data = resp.json()
            break
        except (requests.ConnectionError, requests.Timeout):
            print("timeout/connection")
            time.sleep(backoff + random.uniform(0, backoff * 0.5))
            backoff = min(backoff * 2, 60)
        except Exception as e:
            print("Exception", e, image_url, image_filename)
            return {}
    else:
        print("exhausted retries")
        return {}

    link = data.get("essential", {}).get("link")
    license = data.get("essential", {}).get("license", {}).get("title")
    author = data.get("essential", {}).get("credit")
    return dict(link=link, license=license, author=author)


with open(CSV_ARTICLE_FILE_WITH_SENTENCE_SPLITS, newline="") as f:
     rows = list(csv.DictReader(f))


url_to_attribution: dict[str, dict] = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_attribution, row['image_url']): row['image_url'] for row in rows}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Fetching attribution"):
        url = futures[fut]
        url_to_attribution[url] = fut.result()


fieldnames = list(rows[0].keys()) + ["image_license", "image_credit", "image_attribution_url"]
with open(CSV_ARTICLE_FILE_WITH_ATTRIB, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in rows:
        attribution_info = url_to_attribution.get(r["image_url"], {})
        r['image_license'] = attribution_info.get('license')
        r['image_credit'] = attribution_info.get('author')
        r['image_attribution_url'] = attribution_info.get('link')                
        writer.writerow(r)

Fetching attribution:   0%|          | 0/99930 [00:00<?, ?it/s]

Exception 404 Client Error: Not Found for url: https://en.wikipedia.org/w/rest.php/attribution/v0-beta/pages/File:Mason_Thames_Black_Phone_2_premiere_%282025%29.jpg/signals https://upload.wikimedia.org/wikipedia/commons/f/f3/Mason_Thames_Black_Phone_2_premiere_%282025%29.jpg Mason_Thames_Black_Phone_2_premiere_%282025%29.jpg
Exception 404 Client Error: Not Found for url: https://en.wikipedia.org/w/rest.php/attribution/v0-beta/pages/File:ClavicularNov2025.jpg/signals https://upload.wikimedia.org/wikipedia/commons/8/83/ClavicularNov2025.jpg ClavicularNov2025.jpg
Exception 404 Client Error: Not Found for url: https://en.wikipedia.org/w/rest.php/attribution/v0-beta/pages/File:Russell_Wilson_Giants_080825.jpg/signals https://upload.wikimedia.org/wikipedia/commons/b/b7/Russell_Wilson_Giants_080825.jpg Russell_Wilson_Giants_080825.jpg
Exception 404 Client Error: Not Found for url: https://en.wikipedia.org/w/rest.php/attribution/v0-beta/pages/File:Anna_Nicole_smith_2005.jpg/signals https://upl